# 예제 franka_ex10: FR3 OMPL 플래너 비교

같은 목표 조인트 자세를 여러 OMPL 플래너에게 시켜 보고, 계획 시간/포인트수/궤적 모양을 비교한다.

**6-DOF 예제와 다른 점**
- 7-DOF 라 자유도 추가 → 동일 목표라도 플래너마다 더 다양한 경로가 나올 수 있다
- 끝단 링크 `fr3_hand_tcp`, frame `fr3_link0`
- 비교 목표: FR3 의 큼지막한 자세 변화 (joint1=+90°, joint2=-30°, joint4=-150°…) — 각 플래너가 충분히 큰 변위에서 차이를 드러내도록
- `home` 자세 없음 → 매 플래너 비교 사이마다 `ready` 로 리셋
- `use_sim_time=True`

**학습 내용**
- `MotionPlanRequest.planner_id` 파라미터
- 플래너 종류: RRTConnect / RRT / PRM / EST / KPIECE 의 특성
- 같은 trajectory 를 FK 로 끝단 좌표 시퀀스로 변환해 RViz LINE_STRIP 으로 누적 비교

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/planner_paths` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/planner_paths'

## 2. 초기화

In [ ]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from visualization_msgs.msg import MarkerArray, Marker
from std_msgs.msg import ColorRGBA
from geometry_msgs.msg import Point, Vector3

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex10_planners_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex10 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

## 3. 서버 준비

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup / Execute / FK 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

In [ ]:
from moveit_msgs.action import ExecuteTrajectory

execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
if not execute_client.wait_for_server(timeout_sec=15.0):
    raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')

def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

In [ ]:
from moveit_msgs.srv import GetPositionFK
from moveit_msgs.msg import RobotState

fk_client = node.create_client(GetPositionFK, 'compute_fk')
fk_client.wait_for_service(timeout_sec=10.0)

def trajectory_to_ee_path(trajectory, max_points: int = 60):
    '''RobotTrajectory → fr3_hand_tcp 의 base 기준 좌표 리스트.'''
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            p = resp.pose_stamped[0].pose.position
            pts.append((p.x, p.y, p.z))
    return pts

## 6. 플래너 스타일 + 누적 마커

각 플래너 경로를 다른 색의 LINE_STRIP 으로 RViz 에 같이 띄운다.

In [ ]:
PLANNER_STYLES = {
    'RRTConnect': {'color': ColorRGBA(r=1.0, g=0.2, b=0.2, a=0.95), 'width': 0.005},
    'RRT':        {'color': ColorRGBA(r=0.2, g=0.8, b=0.2, a=0.95), 'width': 0.005},
    'PRM':        {'color': ColorRGBA(r=0.2, g=0.4, b=1.0, a=0.95), 'width': 0.005},
    'EST':        {'color': ColorRGBA(r=1.0, g=0.6, b=0.0, a=0.95), 'width': 0.005},
    'KPIECE':     {'color': ColorRGBA(r=0.7, g=0.2, b=0.9, a=0.95), 'width': 0.005},
}

_markers = MarkerArray()
_marker_id = {'next': 0}
def _nid():
    _marker_id['next'] += 1
    return _marker_id['next']

def add_path_marker(planner_id: str, ee_points, planner_idx: int):
    if not ee_points:
        return
    style = PLANNER_STYLES.get(planner_id, {
        'color': ColorRGBA(r=0.5, g=0.5, b=0.5, a=0.9), 'width': 0.004,
    })
    stamp = node.get_clock().now().to_msg()

    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = f'path_{planner_id}'
    line.id = _nid()
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = style['width']
    line.color = style['color']
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_points]
    _markers.markers.append(line)

    end = Marker()
    end.header.frame_id = REFERENCE_FRAME
    end.header.stamp = stamp
    end.ns = f'end_{planner_id}'
    end.id = _nid()
    end.type = Marker.SPHERE
    end.action = Marker.ADD
    end.pose.position = Point(x=ee_points[-1][0], y=ee_points[-1][1], z=ee_points[-1][2])
    end.pose.orientation.w = 1.0
    end.scale = Vector3(x=0.025, y=0.025, z=0.025)
    end.color = style['color']
    _markers.markers.append(end)

    label = Marker()
    label.header.frame_id = REFERENCE_FRAME
    label.header.stamp = stamp
    label.ns = f'label_{planner_id}'
    label.id = _nid()
    label.type = Marker.TEXT_VIEW_FACING
    label.action = Marker.ADD
    mid = ee_points[len(ee_points)//2]
    label.pose.position = Point(x=mid[0], y=mid[1] + 0.03 * (planner_idx - 2), z=mid[2] + 0.04)
    label.pose.orientation.w = 1.0
    label.scale.z = 0.04
    label.color = style['color']
    label.text = planner_id
    _markers.markers.append(label)
    marker_pub.publish(_markers)

## 7. 시나리오

**비교 목표**: FR3 가 좌측 깊숙이 팔을 뻗고 손목을 비튼 자세로 큰 변위 이동.
RRTConnect 가 가장 빠르고 안정적, EST/KPIECE 는 좁은 공간/고차원에서 차이가 두드러진다.

In [ ]:
target_joints = {
    'fr3_joint1':  math.radians(90),    # 좌측 90°
    'fr3_joint2':  math.radians(-30),   # 어깨 살짝 숙임
    'fr3_joint3':  math.radians(0),
    'fr3_joint4':  math.radians(-150),  # 팔꿈치 깊게 굽힘
    'fr3_joint5':  math.radians(60),    # 손목 비틀기
    'fr3_joint6':  math.radians(60),
    'fr3_joint7':  math.radians(45),
}

planners = [
    {'id': 'RRTConnect', 'desc': '양방향 RRT — 빠르고 안정적 (기본)'},
    {'id': 'RRT',        'desc': '단방향 RRT — 더 단순, 느릴 수 있음'},
    {'id': 'PRM',        'desc': 'Probabilistic Roadmap — 다중 쿼리에 유리'},
    {'id': 'EST',        'desc': 'Expansive Space Trees — 좁은 공간에서 강함'},
    {'id': 'KPIECE',     'desc': '투영 기반 — 고차원 공간에 효과적'},
]

## 8. 각 플래너 비교 실행

각 플래너에 대해:
1. `ready` 로 리셋
2. `plan_to_joint_goal(target_joints, planner_id=...)` — 계획만 (실행 X)
3. trajectory 를 FK 로 끝단 경로로 변환 → RViz 누적
4. `execute_trajectory(traj)` — 실제로도 한 번 실행


In [ ]:
import time

results = []
for i, planner in enumerate(planners):
    node.get_logger().info(f"\n--- 플래너: {planner['id']} ---")
    node.get_logger().info(f"  설명: {planner['desc']}")

    # ready 로 리셋 (RRTConnect 사용 — 비교 대상 아님)
    go_to_joint_goal(ready_target, vel=0.4, acc=0.4)
    time.sleep(0.5)

    # 비교 대상 플래너로 계획만
    t0 = time.time()
    ok, traj = plan_to_joint_goal(
        target_joints, vel=0.3, acc=0.3,
        planner_id=planner['id'], plan_time=10.0,
    )
    plan_time = time.time() - t0

    n_pts = 0
    if ok and traj is not None:
        n_pts = len(traj.joint_trajectory.points)
        ee_pts = trajectory_to_ee_path(traj)
        if ee_pts:
            add_path_marker(planner['id'], ee_pts, i)
            node.get_logger().info(f'  계획 성공! {plan_time:.3f}s, {n_pts} 포인트, EE 경로 {len(ee_pts)}점 마커')
        execute_trajectory(traj)
    else:
        node.get_logger().warn(f'  계획 실패! ({plan_time:.3f}s)')

    results.append({'planner': planner['id'], 'success': ok,
                    'plan_time': plan_time, 'n_pts': n_pts})
    time.sleep(0.5)

# 결과 요약 출력
node.get_logger().info('\n' + '=' * 60)
node.get_logger().info('  플래너 비교 결과')
node.get_logger().info('=' * 60)
for r in results:
    status = 'O' if r['success'] else 'X'
    node.get_logger().info(
        f"  {r['planner']:<12} {status:>3}  {r['plan_time']:>6.2f}s  {r['n_pts']:>4} pts"
    )

## 9. ready 복귀

In [ ]:
go_to_joint_goal(ready_target)
node.get_logger().info('=== franka_ex10 완료! ===')

## 10. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass